In [7]:
import pandas as pd
import geopandas as gpd


# Keep only regular bus routes: the ones in the ridership data
ridership_routes = pd.read_csv(processed + r'\ridership_route_year.csv', dtype={'route_id': str})['route_id'].unique()

before = stops_gdf['route_id'].nunique()
stops_gdf = stops_gdf[stops_gdf['route_id'].isin(ridership_routes)]
print(before, '->', stops_gdf['route_id'].nunique(), 'routes')

processed = r'C:\Users\sahil\projects\boston-mbta-ridership\data\processed'

route_stops = pd.read_csv(processed + r'\route_stops.csv', dtype={'route_id': str, 'stop_id': str})

# Turn latitude/longitude into map points
stops_gdf = gpd.GeoDataFrame(route_stops,
                             geometry=gpd.points_from_xy(route_stops['stop_lon'], route_stops['stop_lat']),
                             crs='EPSG:4326')     # EPSG:4326 = latitude/longitude

# Switch to Massachusetts' map projection, measured in meters, so distances work
stops_gdf = stops_gdf.to_crs('EPSG:26986')
print(len(stops_gdf), 'route-stop points')

341 -> 148 routes
12313 route-stop points


In [8]:
quarter_mile = 402     # meters

# A circle around every stop...
stops_gdf['geometry'] = stops_gdf.buffer(quarter_mile)

# ...merged into one shape per route
route_areas = stops_gdf.dissolve(by='route_id', as_index=False)[['route_id', 'geometry']]
route_areas['route_area_m2'] = route_areas.area

print(len(route_areas), 'route areas')

341 route areas


In [9]:
# ACS counts that get split by area (percentages are recalculated from these afterward)
count_cols = ['pop_total', 'pop_nh_white', 'pov_universe', 'pov_below', 'households', 'hh_no_vehicle',
              'workers', 'workers_transit', 'commuters', 'commute_45_59min', 'commute_60_89min',
              'commute_90plus_min', 'occupied_units', 'renter_units', 'renter_hh', 'rent_30_35pct',
              'rent_35_40pct', 'rent_40_50pct', 'rent_50plus_pct', 'labor_force', 'unemployed']

def route_demographics(tract_file, acs_period):
    """Demographics of residents within a quarter mile of each route."""
    tracts = gpd.read_file(tract_file).to_crs('EPSG:26986')
    tracts['tract_area_m2'] = tracts.area

    # Cut each route area into pieces along tract boundaries
    pieces = gpd.overlay(route_areas, tracts, how='intersection')

    # Share of each tract that falls inside the route area
    pieces['weight'] = pieces.area / pieces['tract_area_m2']

    # Scale each tract's counts by that share
    for col in count_cols:
        pieces[col] = pieces[col] * pieces['weight']

    # Median income can't be added up, so take a population-weighted average of tract medians
    pieces['income_x_pop'] = pieces['median_hh_income'] * pieces['pop_total']
    pieces['pop_with_income'] = pieces['pop_total'].where(pieces['median_hh_income'].notna(), 0)
    pieces['piece_area_m2'] = pieces.area

    # Add up the pieces for each route
    t = pieces.groupby('route_id', as_index=False)[count_cols + ['income_x_pop', 'pop_with_income',
                                                                 'piece_area_m2']].sum()

    # Turn totals into percentages
    result = pd.DataFrame({'route_id': t['route_id'], 'acs_period': acs_period})
    result['residents']         = t['pop_total'].round(0)
    result['median_income_est'] = (t['income_x_pop'] / t['pop_with_income']).round(0)
    result['pct_poverty']       = t['pov_below'] / t['pov_universe'] * 100
    result['pct_no_vehicle']    = t['hh_no_vehicle'] / t['households'] * 100
    result['pct_minority']      = (1 - t['pop_nh_white'] / t['pop_total']) * 100
    result['pct_transit']       = t['workers_transit'] / t['workers'] * 100
    result['pct_long_commute']  = (t['commute_45_59min'] + t['commute_60_89min'] + t['commute_90plus_min']) / t['commuters'] * 100
    result['pct_renters']       = t['renter_units'] / t['occupied_units'] * 100
    result['pct_rent_burdened'] = (t['rent_30_35pct'] + t['rent_35_40pct'] + t['rent_40_50pct'] + t['rent_50plus_pct']) / t['renter_hh'] * 100
    result['unemployment_rate'] = t['unemployed'] / t['labor_force'] * 100

    # How much of the route's walking area lies inside Boston (our ACS only covers Boston)
    result = result.merge(route_areas[['route_id', 'route_area_m2']], on='route_id')
    result['pct_area_in_boston'] = t['piece_area_m2'].values / result['route_area_m2'] * 100

    return result.drop(columns='route_area_m2').round(1)

In [10]:
demo24 = route_demographics(processed + r'\boston_tracts_2024.geojson', '2020-2024')
demo19 = route_demographics(processed + r'\boston_tracts_2019.geojson', '2015-2019')

route_area_demographics = pd.concat([demo19, demo24], ignore_index=True)
route_area_demographics['fare_free'] = route_area_demographics['route_id'].isin(['23', '28', '29'])

print(route_area_demographics.shape)
route_area_demographics[route_area_demographics['route_id'].isin(['23', '28', '29', '22', '1'])]

(452, 14)


,route_id,acs_period,residents,median_income_est,pct_poverty,pct_no_vehicle,pct_minority,pct_transit,pct_long_commute,pct_renters,pct_rent_burdened,unemployment_rate,pct_area_in_boston,fare_free
0,1,2015-2019,39927.0,54613.0,27.2,52.5,52.2,30.5,15.7,77.3,50.7,8.9,51.4,False
26,22,2015-2019,72297.0,38561.0,31.4,44.2,77.8,37.9,25.1,78.7,55.0,10.9,89.5,False
27,23,2015-2019,55899.0,39936.0,28.2,41.4,84.7,37.0,26.8,72.8,57.0,12.5,94.9,True
32,28,2015-2019,69296.0,35322.0,31.2,44.0,82.6,35.4,26.3,79.2,56.6,12.3,89.9,True
33,29,2015-2019,46596.0,38715.0,28.9,39.5,92.9,40.5,31.4,77.6,56.6,10.1,82.7,True
226,1,2020-2024,39518.0,72076.0,27.7,52.5,52.7,22.9,14.4,76.9,56.5,7.5,51.4,False
252,22,2020-2024,72759.0,56874.0,27.1,41.9,77.8,28.0,24.1,77.8,52.8,9.5,89.5,False
253,23,2020-2024,58015.0,54927.0,23.5,36.9,84.7,29.6,27.1,72.1,54.9,9.7,94.9,True
258,28,2020-2024,68680.0,50708.0,27.2,41.4,82.1,28.7,24.8,76.7,56.5,11.1,90.1,True
259,29,2020-2024,44247.0,55527.0,26.3,37.3,91.1,31.4,33.5,74.8,54.7,11.3,82.9,True


In [11]:
route_area_demographics.to_csv(processed + r'\route_area_demographics.csv', index=False)
print('Saved')

Saved


In [12]:
print(route_area_demographics['route_id'].nunique(), 'routes')
print(sorted(route_area_demographics['route_id'].unique()))

226 routes
['1', '10', '101', '104', '105', '109', '11', '111', '112', '114', '116', '120', '121', '14', '15', '16', '17', '171', '18', '19', '201', '202', '21', '210', '215', '217', '22', '23', '24', '240', '245', '26', '28', '29', '30', '31', '32', '33', '34', '34E', '35', '354', '36', '37', '38', '39', '4', '40', '41', '42', '426', '428', '43', '44', '45', '450', '47', '50', '501', '504', '505', '51', '52', '55', '57', '60', '64', '65', '66', '7', '70', '708', '71', '712', '713', '716', '73', '741', '742', '743', '746', '749', '751', '8', '85', '86', '89', '8993', '9', '90', '91', '92', '93', '95', 'Shuttle-AirportGovernmentCenterExpress', 'Shuttle-AirportGovernmentCenterLocal', 'Shuttle-AirportGovernmentCenterLocalViaGate', 'Shuttle-AirportHaymarket', 'Shuttle-AirportHaymarketExpress', 'Shuttle-AirportHaymarketLocal', 'Shuttle-AirportMaverick', 'Shuttle-AirportWonderland', 'Shuttle-AlewifeParkSt', 'Shuttle-AlewifeParkStreet', 'Shuttle-AndersonWoburnNorthStationExpress', 'Shuttle-An

In [13]:
route_area_demographics[route_area_demographics['route_id'].isin(['23', '28', '29', '22', '1'])]

,route_id,acs_period,residents,median_income_est,pct_poverty,pct_no_vehicle,pct_minority,pct_transit,pct_long_commute,pct_renters,pct_rent_burdened,unemployment_rate,pct_area_in_boston,fare_free
0,1,2015-2019,39927.0,54613.0,27.2,52.5,52.2,30.5,15.7,77.3,50.7,8.9,51.4,False
26,22,2015-2019,72297.0,38561.0,31.4,44.2,77.8,37.9,25.1,78.7,55.0,10.9,89.5,False
27,23,2015-2019,55899.0,39936.0,28.2,41.4,84.7,37.0,26.8,72.8,57.0,12.5,94.9,True
32,28,2015-2019,69296.0,35322.0,31.2,44.0,82.6,35.4,26.3,79.2,56.6,12.3,89.9,True
33,29,2015-2019,46596.0,38715.0,28.9,39.5,92.9,40.5,31.4,77.6,56.6,10.1,82.7,True
226,1,2020-2024,39518.0,72076.0,27.7,52.5,52.7,22.9,14.4,76.9,56.5,7.5,51.4,False
252,22,2020-2024,72759.0,56874.0,27.1,41.9,77.8,28.0,24.1,77.8,52.8,9.5,89.5,False
253,23,2020-2024,58015.0,54927.0,23.5,36.9,84.7,29.6,27.1,72.1,54.9,9.7,94.9,True
258,28,2020-2024,68680.0,50708.0,27.2,41.4,82.1,28.7,24.8,76.7,56.5,11.1,90.1,True
259,29,2020-2024,44247.0,55527.0,26.3,37.3,91.1,31.4,33.5,74.8,54.7,11.3,82.9,True
